# - 체인을 구현하는 문법, LCEL
#### 필요 패키지 설치
- 아래 코드 셀에 명령어를 입력하여 필요한 패키지를 설치합니다.

- 코랩 환경에서는 코드 셀에 `!` 기호를 사용하여 `pip`와 관련된 명령어를 수행하도록 할 수 있습니다.

In [14]:
# 아래에 패키지 설치 명령어 작성하기

# !pip install -U langchain langchain-openai

In [ ]:
# 아래에 패키지 버전 확인을 위한 명령어 작성하기
# pip show 패키지 이름으로 확인 할 수 있음

!pip show langchain
!pip show langchain-openai
!pip show langchain-core

#### 오픈AI API 키를 OS 환경 변수에 입력

- 환경 변수에 API 키를 입력해 둠으로써, 앞으로 이 파일에서는 API 키를 다시 입력할 필요가 없습니다.

In [16]:
from dotenv import load_dotenv

load_dotenv()


True

---

## 1. LCEL 문법 이해
### 1.1 LCEL 체인의 구축 및 실행

#### 프롬프트 정의
- `from_template` 메서드에 프롬프트 정의하기



In [17]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template('{topic}에 대해 쉽게 설명해주세요')


#### LLM 모델 객체 생성
- 오픈 AI의 LLM 모델 불러오기
- API Key는 위에서 환경 변수에 설정하였으므로 별도 과정 필요 없음

In [18]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model='gpt-5-nano')

#### 출력 파서 정의
- 출력 파서는 LLM 모델이 생성한 응답을 적절한 형태로 변환합니다.

In [19]:
from langchain_core.output_parsers import StrOutputParser  # StrOutputParser : 메시지, 모델의 정보를 가지고 질문하면 답변 생성해서 피드백

output_parser = StrOutputParser() # 생성자 클래스

#### 구성 요소 연결
- 파이프 `|` 연산자로 구성 요소를 연결합니다.

In [20]:
chain = prompt | model | output_parser   # chain으로 연결

#### 최종 출력 생성

In [ ]:
response = chain.invoke({'topic':'양자역학'})
print(response)

#### stream 메서드
- 챗GPT처럼 답변이 실시간으로 생성되는 듯 자연스러운 연출을 구현하고 싶다면, stream 메서드를 사용할 수 있습니다.

In [ ]:
for chunk in chain.stream({'topic':'양자역학'}):  # 모델이 보내오는 chunk 단위 데이터를 실시간으로 반환
    print(chunk, end='')

#### 프롬프트에 다수의 변수 정의하기
- {주제}에 대해 {난이도} 수준으로 설명해 달라는 프롬프트를 작성하기
- 각 변수 명은 `topic`과 `level`로 설정

In [ ]:
# 새로운 프롬프트 정의
prompt2 = PromptTemplate.from_template('{topic}에 대해 {level}수준으로 설명해 주세요')

# 새로운 체인 생성
chain2 = prompt2 | model | output_parser

# 체인 실행
response = chain2.invoke({'topic':'인공지능', 'level':'초등학생'})
print(response)


---

## 2. LCEL 구축 시 주의 사항
### 2.1 다중 입력 변수 처리 및 KeyError
- 앞서 두 개의 입력 변수 `topic`과 `level`을 프롬프트에 정의한 경우, `invoke` 메서드를 호출할 때 하나라도 빠트리면 `KeyError`가 발생합니다.

In [ ]:
# 잘못된 호출 (level 변수 누락 -> KeyError 발생)
response_error = chain2.invoke({'topic':'인공지능'})


### 2.2 체인 구성 및 ValueError

- 체인은 반드시 프롬프트(prompt) -> 모델(model) -> 출력 파서(output_parser) 순서로 연결되어야 합니다.

In [ ]:
# 잘못된 체인 구성
chain3 = model | prompt2 | output_parser  # 순서가 바뀜, ValueError 발생
response = chain3.invoke({'topic':'인공지능', 'level':'초등학생'})
print(response)


---

## 3. 결합된 체인(Combined Chain) 구축
#### 첫 번째 체인 생성
- 아직 invoke 메서드를 호출한 적은 없으므로, 이 딕셔너리의 값에는 chain1 객체만 할당
- 즉, 실제로 model을 호출한 적은 없는 것입니다.

In [26]:
# 첫 번째 체인의 프롬프트
prompt1 = PromptTemplate.from_template('{topic}에 대해 {level} 수준으로 설명해 주세요.') # 템플릿 객체 생성

# 첫 번째 체인생성
chain1 = prompt2 | model | output_parser   # StrOutputParse : 출력 파서

# 새로운 딕셔너리 생성
chain1_result_dict = {'chain1_result':chain1}

#### 두 번째 체인 생성
- 이 프롬프트는 새롭게 만든 딕셔너리(chain1_result_dict)의 키(chain1_result)를 입력 변수로 가져야 합니다.

In [27]:
# 두 번째 체인의 프롬프트 
prompt2 = PromptTemplate.from_template('이 정보가 유익한가요? 정보 : {chain1_result}')  # self 검증

# combined_chain 생성
# chain1_result_dict : 두번 째 chain의 입력 프롬프트
# 한번 질문한 내용에 대한 응답을 받은 이후에는 해당 질문을 관리하지 않고 연결이 종료(Http 프로토콜)
# 대화의 맥락을 유지하는 기능 필요
# 첫번 째 chain에 두번 째 chain을 연결
combined_chain = chain1_result_dict | prompt2 | model | output_parser   

#### 결합된 체인 실행 : Combined Chain
- 첫 번째 체인의 프롬프트에 정의된 입력 변수 `topic`과 `level`을 딕셔너리 형태로 전달합니다.

In [ ]:
final_result = combined_chain.invoke({'topic' : '양자역학', 'level': '초등학생'})
print(final_result)

---

## 4. 프롬프트
- 프롬프트는 'prompt'의 의미(무언가를 지시하는 문장) 그대로 AI 모델에게 내리는 지시 또는 명령을 의미합니다.

### 4.1 PromptTemplate.from_template()
- 가장 단순하며 다양한 상황에서 융통성 있게 활용 가능한 방법

In [30]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template('{topic}에 대해 {level} 수준으로 설명해 주세요')
print(prompt)


input_variables=['level', 'topic'] input_types={} partial_variables={} template='{topic}에 대해 {level} 수준으로 설명해 주세요'


### 4.2 PromptTemplate
- from_template 메서드 없이 PromptTemplate를 직접 사용하는 방법

In [37]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate(
    template='{topic}에 대해 {level} 수준으로 설명해 주세요',
    input_variables=['topic', 'level'],
     partial_variables={"topic": "양자역학"}  # 하나의 변수에 특정 값을 defaul값으로 셋팅해서 프롬프트를 고정('{topic}':'양자역학'), 변수는 level 하나만 셋팅
)

print(prompt)

input_variables=['level'] input_types={} partial_variables={'topic': '양자역학'} template='{topic}에 대해 {level} 수준으로 설명해 주세요'


#### input_variables와 partial_variables 확인
- 주의할 점은 partial_variables에 사전 정의된 변수는 partial_variables 속성에서 별도로 확인하여야 합니다.

In [38]:
print(prompt.input_variables)
print(prompt.partial_variables)

['level']
{'topic': '양자역학'}


#### partial_variables 1
- 특정 변수의 값을 사전에 설정하기 위해 사용

In [ ]:
prompt = PromptTemplate(
    template='{year} + {n} + {m} 는 무엇인가요?',
    partial_variables={'year':'2026'}    # default 값 : 'year':'2026'
)

chain = prompt | model | output_parser

chain.invoke({'n':'3','m':'5'})

'2034입니다. 계산: 2026 + 3 = 2029, 2029 + 5 = 2034.'

#### partial_variables 2
- 고정된 값뿐만 아니라 함수도 연결할 수 있습니다.

In [40]:
from datetime import datetime

# 올해 연도를 문자열로 반환하는 함수
def get_current_year():
    return str(datetime.now().year)

prompt = PromptTemplate(
    template='{year} + {n} + {m} 는 무엇인가요?',
    partial_variables={'year': get_current_year}   
)

chain = prompt | model | output_parser

chain.invoke({'n' : '3', 'm' : '5'})

'2034입니다. 계산 과정: 2026 + 3 = 2029, 2029 + 5 = 2034.'

#### partial_variables 주의 사항
- partial_variables에 `n`을 10으로 미리 할당했더라도, `invoke` 메서드 호출 시 새로운 `n` 값을 전달하면 메서드 호출시 전달된 값으로 업데이트되어 프롬프트가 완성됩니다

In [ ]:
prompt = PromptTemplate(
    template='{year} + {n} + {m} 는 무엇인가요?',
    partial_variables={
        'year': '2026',
        'n': '10'
        }   
)

chain = prompt | model | output_parser
chain.invoke({'n' : '3', 'm' : '5'})  # 현재 전달하는 값이 우선 순위가 높다 n : 3 -> 2026 + 3 = 2029
# chain.invoke({'m' : '5'})


'2034입니다. 계산: 2026 + 3 = 2029, 여기에 5를 더하면 2034가 됩니다.'

### 4.3 ChatPromptTemplate
- 이전 대화 내역을 추가함으로써 과거 대화와 연관된 답변을 생성하게 할 수 있게 합니다.

In [43]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate([
        ('system', '당신은 강아지 안구 건강 관련 조언을 해주는 AI 챗봇입니다.'),
        ('human', '안녕하세요! 저희 집 강아지가 눈을 자주 긁어요'),   # human, ai : LangChain 에서 사용하는 프롬프트
        ('ai', '그렇군요. 사용자님의 강아지는 몇 살인가요?'),
        ('human', '{user_input}')
])

chain = prompt | model | output_parser
chain.invoke({'user_input' : '3살이에요'})


'3년이라면 아직 어리진 않지만 눈 긁힘이 지속되면 불편하고 심각한 문제가 있을 수 있어요. 몇 가지 가능성으로는 알레르기, 결막염, 각막의 찰과(미세한 긁힘이나 상처), 건조한 눈, 이물질 등에 의한 자극 등이 있습니다. 특히 눈을 자주 긁는 건 통증이나 불편함의 신호일 수 있어 주의가 필요합니다.\n\n지금 바로 체크해볼 점\n- 어떤 눈인가요? 왼쪽/오른쪽/양쪽 중 어디인가요?\n- 분비물 색과 양은 어떤가요? 투명한가요, 노랗거나 녹색처럼 색이 있나요? 끈적임은 있나요?\n- 눈이 빨개지거나 부어 있나요? 눈을 찌푸리거나 빛에 민감해 하나로 쳐다보는 모습이 있나요?\n- 이물질이 들어갔을 법한 상황이 있었나요? 예: 바람 많은 환경, 모래, 풀빛 등\n- 최근에 다치거나 긁힌 흔적이 있나요? 눈에 직접 상처가 보이면 안쪽 충혈이 보일 수 있습니다\n- 현재 아무 약이나 바르지 않았나요? 사람용 약이나 무책임한 눈약은 절대 주지 마세요\n\n지금 당장 할 수 있는 안전한 조치\n- 눈을 더 긁지 않도록 핸드폰으로 손톱을 다듬고, 가능하면 E-칼라(목걸이형 케이지)로 긁는 행위를 막아 주세요.\n- 눈 주위의 분비물은 깨끗한 물수건이나 깨끗한 물에 적신 거즈로 가볍게 닦아내되 눈 자체를 문지르지 마세요.\n- 직접 약을 바르거나 인간용 의약품을 사용하지 마세요. 특히 점안제는 수의사의 처방 없이 사용하지 마세요.\n\n언제 수의사 방문이 필요할까요\n- 눈이 심하게 빨개지거나 지속적으로 흐르는 눈물/분비물이 있으면\n- 눈을 거의 감고 빛에 반응하지 않거나 심한 찌푸림/통증 징후가 보이면\n- 이물질이 보이거나 눈꺼풀이 안쪽으로 말려들어간(에센트리온) 등의 구조적 이상이 의심되면\n- 24시간 내에 증상이 좋아지지 않으면\n- 타액, 코빵/녹색 분비물, 시력 저하 의심 등 동반 증상이 있으면 즉시\n\n원하시면 제가 더 구체적으로 안내해 드리겠습니다\n- 어느 눈인지(왼쪽/오른쪽)와 분비물 색상, 증상 지속 시간\n- 반려견의 품종이나 털 관리 상태, 최근

---

## 5. 모델
#### 다양한 모델 사용하기
  - langchain-openai는 위에서 설치하였으므로 건너뜀

  - 아래는 예시로 코드로 별도로 실행하지 않습니다.

In [ ]:
# !pip install -U langchain-anthropic

In [ ]:
# 모델 변경이 간단, claude 모델을 위한 API Key를 발급받지 않았으므로, 이번 실습에서는 사용 안함
# from langchain_anthropic import ChatAnthropic
# model = ChatAnthropic(model='claude-3-sonnet')


#### 오픈AI 모델 하이퍼파라미터

- model: 오픈AI에서 제공하는 여러 LLM 모델 중 특정 모델을 지정
- temperature: 생성되는 텍스트의 다양성을 조절하는 매개변수
    - 낮은 값(예: 0.2)은 더 결정적이고 일관된 출력을 생성
    - 높은 값(예: 0.8)은 더 창의적이고 다양성이 높은 출력을 생성
    - 이 값이 높을수록 모델이 더 다양한 단어와 구문을 선택하도록 유도하여 창의적인 결과를 도출
- max_tokens: 모델이 생성할 수 있는 최대 토큰 수를 지정
- max_retries: 모델 호출이 실패할 경우 재시도 횟수를 지정


In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model='gpt-4o-mini',  # 필수 파라미터
    # temperature=0.2,  # 사전적(정석적) 개념의 답변
    temperature=0.9,    # 창의적 개념의 답변
    max_tokens=2048,
    max_retries=2 
)

#### 실습
- 생성한 모델의 하이퍼파라미터를 조정하며 다양한 답변 받아보기

In [50]:
prompt = PromptTemplate.from_template('{topic}에 대해 쉽게 설명해 주세요')

chain = prompt | model | output_parser

chain.invoke({'topic': '사랑'})

'사랑은 사람들 사이의 깊고 특별한 감정입니다. 누군가를 아끼고, 이해하며, 그 사람의 행복을 바라는 마음을 포함합니다. 사랑은 다양한 형태가 있는데, 가족 간의 사랑, 친구 간의 사랑, 연인 간의 사랑 등이 있습니다. \n\n사랑은 기쁨과 행복을 줄 뿐 아니라, 때로는 슬픔이나 아픔을 동반하기도 합니다. 사랑하는 사람과 함께하는 시간은 소중하고, 서로의 어려움을 함께 나누면서 더 깊은 유대감을 형성하게 됩니다. \n\n결국 사랑은 서로를 존중하고 지지하며, 함께 성장하는 소중한 경험이라고 할 수 있습니다.'

---

## 6. 출력 파서
#### 출력 파서 없이 코드 실행하기
- 아래 코드를 실행하여 결과를 확인합니다.

In [52]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

prompt = PromptTemplate.from_template('{topic}에 대해 쉽게 설명해 주세요')

model = ChatOpenAI(
    model='gpt-5-nano',
    max_tokens=2048
)

chain = prompt | model
chain.invoke({'topic' : '인공지능'})

AIMessage(content='인공지능은 컴퓨터가 사람처럼 배우고, 판단하고, 문제를 해결하도록 만드는 기술이에요.\n\n쉽게 이해하는 비유\n- 컴퓨터를 학생처럼 생각하고, 데이터는 책과 연습문제, 규칙은 시험 문제 유형이라고 보면 돼요.\n- 많이 배우고 연습하면 점점 더 똑똑해져서 우리가 시켜 주는 일을 더 잘해요.\n\n작동 방식의 핵심 아이디어\n- 데이터에서 배우는 것: 컴퓨터가 예시를 보고 패턴을 찾아내고, 그 패턴으로 새롭게 주어진 문제를 해결해요.\n- 규칙을 직접 쓰는 대신, 데이터를 통해 규칙을 스스로 추론하는 방식이에요.\n\n가장 흔한 유형과 예\n- 지도학습: 정답(정답 예시)이 주어지고, 이를 따라 새 데이터를 맞히는 학습이에요. 예) 이메일 스팸 여부 판단, 사진에서 고양이/개 구분.\n- 비지도학습: 정답이 없고, 데이터 안의 구조를 찾아내는 학습이에요. 예) 사진 속 비슷한 물건끼리 묶기(클러스터링).\n- 강화학습: 행동의 결과로 보상을 받으며 최적의 행동을 배우는 방식이에요. 예) 게임에서 최적의 전략 찾기, 로봇이 목표에 맞게 움직이도록 학습하기.\n- 일상 예시: 스마트폰의 음성인식, 음악/영상 추천 시스템, 번역 서비스, 자율주행 자동차.\n\n인공지능이 다루는 일과 아닌 것\n- 잘하는 일: 대량의 데이터를 빠르게 분석하고 패턴을 찾아 의사결정이나 예측을 돕는 것.\n- 아직 어려운 점: 사람처럼 넓은 범위의 일반 지능(상황을 다 이해하고 창의적으로 대처하는 능력), 감정, 직관, 공통 상식은 부족하고 편향이나 잘못된 데이터에 영향을 받기 쉽습니다.\n\n주의점과 건강한 활용\n- 데이터 편향으로 잘못된 결론이 나올 수 있음.\n- 항상 옳지 않으며, 중요한 결정은 사람의 판단이 함께 필요.\n- 개인정보 보호와 보안, 투명성(왜 그런 결과가 나왔는지 이해하는 것)이 중요.\n\n간단하게 시작하는 방법\n- 일상에서 사용하는 AI 도구를 한두 가지 써보고, 어떤 데이터로 어떻게 작동하는지 관찰해 보기.\n- 기본 

---

### CommaSeparatedOutputParser, JsonOutputParser 알아보기
#### 프롬프트 구성 예시

In [53]:
template = """
당신은 K-Pop 정보를 제공하는 AI입니다. 그룹의 멤버 이름을 말해주세요.
K-Pop group : {name}  

FORMAT:
{format}
"""

prompt = PromptTemplate(
    template=template,
    partial_variables={'format': 'JSON 형태로 답변하세요'},
)

#### get_format_instructions 메서드와 활용 예시

In [57]:
from langchain_core.output_parsers import JsonOutputParser

output_parser = JsonOutputParser()
format_instructions = output_parser.get_format_instructions()

# print(format_instructions)  # Return a Json object

prompt = PromptTemplate(
    template=template,
    partial_variables={'format': format_instructions}
)

print(prompt)

input_variables=['name'] input_types={} partial_variables={'format': 'Return a JSON object.'} template='\n당신은 K-Pop 정보를 제공하는 AI입니다. 그룹의 멤버 이름을 말해주세요.\nK-Pop group : {name}  \n\nFORMAT:\n{format}\n'


---

### 6.1 CommaSeperatedListOutputParser
#### 콤마 구분자 출력 파서
1. 클래스 import

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

# 파서 초기화
output_parser = CommaSeparatedListOutputParser()

2. 프롬프트 불러오기

In [ ]:
format_instructions = output_parser.get_format_instructions()
print(output_parser)

prompt = PromptTemplate(
    template=template,
    partial_variables={'format': format_instructions}
)

print(prompt)


input_variables=['name'] input_types={} partial_variables={'format': 'Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`'} template='\n당신은 K-Pop 정보를 제공하는 AI입니다. 그룹의 멤버 이름을 말해주세요.\nK-Pop group : {name}  \n\nFORMAT:\n{format}\n'


3. 프롬프트 완성

In [60]:
template = """
당신은 K-Pop 정보를 제공하는 AI입니다. 그룹의 멤버 이름을 말해주세요.
K-Pop group : {name}  

FORMAT:
{format}
"""

prompt = PromptTemplate(
    template=template,
    partial_variables={'format': format_instructions},
)

4. 모델 생성

In [61]:
model = ChatOpenAI(model='gpt-5-nano')

5. 체인 연결과 실행

In [ ]:
chain = prompt | model | output_parser

result = chain.invoke({'name': '블랙핑크'})  # CommaSeparatedListOutputParser 사용

print('출력 타입: ', type(result))  # <class 'list'>
print('출력 결과: ', result)  # ['Jisoo', 'Jennie', 'Rosé', 'Lisa']

출력 타입:  <class 'list'>
출력 결과:  ['Jisoo', 'Jennie', 'Rosé', 'Lisa']


---

### 6.3 JsonOutputParser
1. 클래스 import

In [77]:

from langchain_core.output_parsers import JsonOutputParser

output_parser= JsonOutputParser()


2. 프롬프트 불러오기

In [79]:
format_instructions = output_parser.get_format_instructions()
print(format_instructions)


Return a JSON object.


3. 프롬프트 완성

In [80]:
template = """
당신은 K-Pop 정보를 제공하는 AI입니다. 그룹의 멤버 이름을 말해주세요.
K-Pop group : {name}  

FORMAT:
{format}
"""
prompt = PromptTemplate(
    template=template,
    partial_variables={'format': format_instructions},
)

print(prompt)
 

input_variables=['name'] input_types={} partial_variables={'format': 'Return a JSON object.'} template='\n당신은 K-Pop 정보를 제공하는 AI입니다. 그룹의 멤버 이름을 말해주세요.\nK-Pop group : {name}  \n\nFORMAT:\n{format}\n'


4. 체인 연결과 실행
- 실행할 때마다 JSON의 구조가 달라지는 문제가 있음
- 코드 작성 후, 여러번 실행하여 동일한 결과가 나오는지 확인합니다.

In [ ]:
chain = prompt | model | output_parser
result = chain.invoke({'name': 'BTS'})

print('출력 타입: ', type(result))  # <class 'dict'>
print('출력 결과: ', result)  # 실행할 때마다 JSON 구조가 달라짐 -> combined chain 구성 시 문제 발생

# {'group': 'BTS', 'members': ['Jin', 'Suga', 'J-Hope', 'RM', 'Jimin', 'V', 'Jungkook']}
# {'group': 'BTS', 'members': [{'stage_name': 'RM', 'real_name': 'Kim Nam-joon'}, {'stage_name': 'Jin', 'real_name': 'Kim Seok-jin'}, {'stage_name': 'Suga', 'real_name': 'Min Yoon-gi'}, {'stage_name': 'J-Hope', 'real_name': 'Jung Ho-seok'}, {'stage_name': 'Jimin', 'real_name': 'Park Ji-min'}, {'stage_name': 'V', 'real_name': 'Kim Tae-hyung'}, {'stage_name': 'Jungkook', 'real_name': 'Jeon Jung-kook'}

출력 타입:  <class 'dict'>
출력 결과:  {'group': 'BTS', 'members': ['RM', 'Jin', 'Suga', 'J-Hope', 'Jimin', 'V', 'Jungkook']}


#### JSON 구조 직접 지정하기

In [99]:
template = """
당신은 K-Pop 정보를 제공하는 AI입니다. 그룹의 멤버 이름을 말해주세요.
K-pop group : {name}
FORMAT :
{format}
"""
prompt = PromptTemplate(
    template=template,
    partial_variables={
        "format": """
        Return a JSON object like this:
        {
        "answer" : "질문에 대한 답변을 작성합니다",
        "source" : "답변 생성에 활용한 출처를 작성합니다."
        }
    """
    },
)
chain = prompt | model | output_parser
result = chain.invoke({"name" : "블랙핑크"})
print('출력 결과 : ', result)

출력 결과 :  {'answer': '지수, 제니, 로제, 리사', 'source': 'https://ko.wikipedia.org/wiki/블랙핑크, https://en.wikipedia.org/wiki/Blackpink'}


#### chain.invoke({"name" : "블랙핑크"})`를 반복적으로 실행해 보며 결과를 확인해 봅시다.

In [100]:
result = chain.invoke({'name': '블랙핑크'})
print('출력 결과:', result)

출력 결과: {'answer': '지수, 제니, 로제, 리사 (영어 표기: Jisoo, Jennie, Rosé, Lisa)', 'source': '블랙핑크 공식 멤버 정보 및 위키피디아의 블랙핑크 페이지'}


---

### 7. 랭체인 허브
- 사전 정의된 프롬프트 불러오기

In [ ]:
# Create a LANGSMITH_API_KEY in Settings > API Keys
import os
from dotenv import load_dotenv
from langsmith import Client

load_dotenv()

api_key = os.getenv('LANGSMITH_API_KEY')
client = Client(api_key=api_key)

prompt = client.pull_prompt("hardkothari/prompt-maker",
                             include_model=True,
                             dangerously_pull_public_prompt=True)

print(prompt)   # 객체 확인

input_variables=['lazy_prompt', 'task'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'hardkothari', 'lc_hub_repo': 'prompt-maker', 'lc_hub_commit_hash': 'c5db8eeefa7be4862a9599b759608dd10ee53f53910838f69abb5ab31c257c2d'} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an expert Prompt Writer for Large Language Models.\n\n'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['lazy_prompt', 'task'], input_types={}, partial_variables={}, template='Your goal is to improve the prompt given below for {task} :\n--------------------\n\nPrompt: {lazy_prompt}\n\n--------------------\n\nHere are several tips on writing great prompts:\n\n-------\n\nStart the prompt by stating that it is an expert in the subject.\n\nPut instructions at the beginning of the prompt and use ### or to separate the instruction and context \n\nBe specific, descriptive and as deta

- 체인 구성

In [119]:
chain = prompt | ChatOpenAI() | StrOutputParser()  # chain = prompt | model | output_parser

task = """ 
반드시 한글로 작성되어야 합니다.
사용자의 질문을 읽고, 핵심 키워드를 파악해서 전문 지식이 있는 사람의 질문으로 변경해 주세요.
더 체계적이고 단계적인 질문이 될 수 있도록 변경하세요.
사용자의 질문에서 벗어나서는 안됩니다.
"""

lazy_prompt = '강아지 눈이 좀 이상해요'

improved_prompt = chain.invoke({'task': task, 'lazy_prompt': lazy_prompt})  # hardkothari/prompt-maker 프롬프트

print(improved_prompt)

한국 축산 전문가로서, 다음과 같은 전문적인 질문을 작성해 주실 수 있을까요?

반려견의 눈이 이상하다고 주장하면서 무슨 증상이 있는지 자세히 설명해 주실 수 있을까요?


- 다른 질문으로 테스트하기